## Introduction to Modin

In [1]:
!pip install uv -q


[notice] A new release of pip is available: 23.2.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [2]:
!uv pip install "modin[all]"

Using Python 3.12.1 environment at: /workspaces/TechCatalyst_DE_2025v2/dev2
Resolved 122 packages in 1.24s                                       
⠙ Preparing packages... (0/93)                                                  
⠙ Preparing packages... (0/93)-------------     0 B/52.16 KiB           
⠙ Preparing packages... (0/93)-------------     0 B/52.16 KiB           
mistune              ------------------------------     0 B/52.16 KiB
⠙ Preparing packages... (0/93)-------------     0 B/57.32 KiB           
mistune              ------------------------------     0 B/52.16 KiB
⠙ Preparing packages... (0/93)------------- 14.87 KiB/57.32 KiB         
mistune              ------------------------------     0 B/52.16 KiB
⠙ Preparing packages... (0/93)------------- 14.87 KiB/57.32 KiB         
mistune              ------------------------------     0 B/52.16 KiB
⠙ Preparing packages... (0/93)------------- 14.87 KiB/57.32 KiB         
mistune              ------------------------------    

In [3]:
!uv pip install "bokeh>=3.1.0"

Using Python 3.12.1 environment at: /workspaces/TechCatalyst_DE_2025v2/dev2
Resolved 16 packages in 235ms                                        
⠙ Preparing packages... (0/5)                                                   
⠙ Preparing packages... (0/5)--------------     0 B/88.27 KiB           
⠙ Preparing packages... (0/5)--------------     0 B/88.27 KiB           
xyzservices          ------------------------------     0 B/88.27 KiB
⠙ Preparing packages... (0/5)--------------     0 B/354.10 KiB          
xyzservices          ------------------------------     0 B/88.27 KiB
⠙ Preparing packages... (0/5)--------------     0 B/354.10 KiB          
xyzservices          ------------------------------     0 B/88.27 KiB
⠙ Preparing packages... (0/5)--------------     0 B/354.10 KiB          
xyzservices          ------------------------------     0 B/88.27 KiB
contourpy            ------------------------------     0 B/354.10 KiB
⠙ Preparing packages... (0/5)--------------     0 B/6.34 

## Using Pandas

In [7]:
s3file = 's3://techcatalyst-raw/yellow_tripdata_2024-01.parquet'

In [5]:
!uv pip install dotenv

Using Python 3.12.1 environment at: /workspaces/TechCatalyst_DE_2025v2/dev2
Resolved 2 packages in 88ms                                          
⠙ Preparing packages... (0/2)                                                   
⠙ Preparing packages... (0/2)--------------     0 B/20.07 KiB           
⠙ Preparing packages... (0/2)--------------     0 B/20.07 KiB           
dotenv               ------------------------------     0 B/1.85 KiB
⠙ Preparing packages... (0/2)--------------     0 B/20.07 KiB           
dotenv               ------------------------------ 1.85 KiB/1.85 KiB
⠙ Preparing packages... (0/2)--------------     0 B/20.07 KiB           
⠙ Preparing packages... (0/2)--------------     0 B/20.07 KiB           
⠙ Preparing packages... (0/2)---------- 16.00 KiB/20.07 KiB         
Prepared 2 packages in 13ms                                                  
░░░░░░░░░░░░░░░░░░░░ [0/2] Installing wheels...                                 warning: Failed to hardlink files; falling

In [6]:
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
%%time

import pandas as pd

df = pd.read_parquet(s3file)

# 2. Calculate trip duration in minutes
df["trip_duration_min"] = (
    (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60
)

# 3. Filter trips longer than 5 minutes
df_filtered = df[df["trip_duration_min"] > 25]

# 4. Group by payment_type and get average fare
result = df_filtered.groupby("payment_type")["fare_amount"].mean()

result.head()



CPU times: user 2.23 s, sys: 1.14 s, total: 3.37 s
Wall time: 4.58 s


payment_type
0    40.175634
1    49.081101
2    49.536832
3    23.084967
4     4.193730
Name: fare_amount, dtype: float64

## Using Modin

__Small to Medium Data Sizes__. 
Modin adds distributed execution overhead. For DataFrames that fit into memory (typical on laptops/workstations), this overhead can outweigh any parallel gains. Each operation must be scheduled across multiple partitions, even if you have only a few cores or your data is “small.”

In [9]:
import modin.pandas as pd
import os
os.environ["MODIN_ENGINE"] = "dask"

In [11]:
!uv pip install s3fs

Using Python 3.12.1 environment at: /workspaces/TechCatalyst_DE_2025v2/dev2
Resolved 20 packages in 306ms                                        
⠙ Preparing packages... (0/13)                                                  
⠙ Preparing packages... (0/13)-------------     0 B/347.28 KiB          
⠙ Preparing packages... (0/13)------------- 14.91 KiB/347.28 KiB        
⠙ Preparing packages... (0/13)------------- 14.91 KiB/347.28 KiB        
aiobotocore          ------------------------------     0 B/82.33 KiB
⠙ Preparing packages... (0/13)------------- 14.91 KiB/347.28 KiB        
aiobotocore          ------------------------------ 14.88 KiB/82.33 KiB
⠙ Preparing packages... (0/13)------------- 14.91 KiB/347.28 KiB        
aiobotocore          ------------------------------ 14.88 KiB/82.33 KiB
⠙ Preparing packages... (0/13)------------- 14.91 KiB/347.28 KiB        
aiobotocore          ------------------------------ 14.88 KiB/82.33 KiB
⠙ Preparing packages... (0/13)------------- 14.91

In [12]:
%%time
# 1. Read Parquet
df = pd.read_parquet(s3file)

# 2. Calculate trip duration in minutes
df["trip_duration_min"] = (
    (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60
)

# 3. Filter trips longer than 5 minutes
df_filtered = df[df["trip_duration_min"] > 25]

# 4. Group by payment_type and get average fare
result = df_filtered.groupby("payment_type")["fare_amount"].mean()

result.head()

CPU times: user 1.55 s, sys: 238 ms, total: 1.79 s
Wall time: 7.47 s


payment_type
0    40.175634
1    49.081101
2    49.536832
3    23.084967
4     4.193730
Name: fare_amount, dtype: float64